# ETL & EDA of OECD Regional Well-Being Data

## Objectives

* Write your notebook objective here, for example, "Fetch data from Kaggle and save as raw data", or "engineer features for modelling"


## Inputs

* Write down which data or information you need to run the notebook 

## Outputs

* Write here which files, code or artefacts you generate by the end of the notebook 

## Additional Comments

* If you have any additional comments that don't fit in the previous bullets, please state them here. 



---

## Set Up directories and Import Necessary Libraries

In [2]:
# System and OS related tasks
import sys
import os
import importlib
# Add the project root to Python path
project_root = os.path.abspath('..')
sys.path.insert(0, project_root)

# path to directories
raw_dir = '../data/raw'
processed_dir = '../data/processed'

In [3]:
# Get the necessary libraries
import pandas as pd
import numpy as np

---

# 1.0 Read in the OECD Regional Well-Being Excel Spreadsheet

The OECD Regional Well-Being data is obtained from [https://oecdregionalwellbeing.org/](https://oecdregionalwellbeing.org/). It is a .xlsx file with the following sheets:
* "Home": This is the "landing page" for this report. It has navigation links to exploring regional well-being by topic, viewing trends over time and acessing the well-being indicator values.
* "Score_Last": This sheet contains the latest regional scores for the following eleven well-being indicators 
  1. Income, 
  2. Jobs, 
  3. Housing, 
  4. Education, 
  5. Health, 
  6. Evironment, 
  7. Safety, 
  8. Civic Engagement, 
  9. Access to Services, 
  10. Community and 
  11. Life Satisfaction.
* "Score_Trend": This sheet shows the directional trends for same eleven well-being indicaators listed in "Score_Last" sheet.
* **"Indicator_Last"**: 
  * This sheet is what this project is interested in. 
  * It contains the raw data of the indicator values for the following fifteen well-being indicators:
    1. Disposable income per capita in USD PPP, 
    2. Employment rate, 
    3. Unemployment rate, 
    4. Homicide rate per 100,000 people, 
    5. Life expectancy (years), 
    6. Population with at least secondary education (%age), 
    7. Number of rooms per capita (rooms per person), 
    8. Mortality rate (per 1000 people), 
    9. Voter turnout (%age), 
    10. Households broadband access (%age of household), 
    11. Air quality (PM2.5 in µg/m³), 
    12. Life satisfaction index, 
    13. Satisfaction with housing affordability (%age), 
    14. Internet speed from OECD average (%) and 
    15. Perceived social network support index
* "Source": List the data souces and the kinks for each indicatory across different countries.
* "Reference_year": List the reference years for the lastest data and the baseline year for each indicator of each country.

The "Indicator_Last sheet is made up of 2 main sections:
* the regional data for each country. This is at TL2 level of granularity:

![OECD Regional Wellbeing .xls File: "Indicator_Last" sheet for data at country's TL2 level granularity](../docs/images/01_OECD_Regional_Wellbeing_Indicator_Last_Sheet_TL2.png)

* the data for each country. 

![OECD Regional Wellbeing .xls File: "Indicator_Last" sheet for data at country granularity](../docs/images/01_OECD_Regional_Wellbeing_Indicator_Last_Sheet_Country.png)

## 1.1 Data Granularity

📌 According to the OECD, TL2 (Territorial Level 2) refers to the larger administrative regions within member countries, representing the first administrative tier of subnational government. 

📌 I have visually verified that the data for the UK in this Regional Well-Being file is indeed at TL2 level using the [Office of National Statistics' definition of "International Territorial Levels (ITLs)" ](https://cy.ons.gov.uk/methodology/geography/ukgeographies/eurostat#relationship-of-itl-areas-to-uk-administrative-geographies). 

The "Indicator_Last' last also contain data at different units of measurements:
* The Disposable income per capita is in U$ at Purchasing Power Parity.
* Indicators like Life Satisfaction is an index (0 to 10)
* Indicators like Employment rate and Voter turnouts are in percentages
* Other rate type indicators like Homicide are in per 100,000 people while the Mortality rate is per 1000 people. 
* Air polution is in µg/m³ (mircograms per cubic metre)

‼️ As a note of interest, the "Perceived Social Network Support" indicator was labelled as an index (0 to 10) whilst the values for this indicator are more than 10. I have verified that subjective indicators such as this, are calculated bu OECD based on pooled data from the GA;;ip World Poll. This indicator is measured as a share of respondents who report having friends or relatives they can count on in times of need (see [definition of indicators for Well-Being Data](https://github.com/mmartamagna/OECD-Regional-Well-being-dataset)).

The variey of units of measurements reflect the different dimensions of well-being and there is no need to standarise them as they will be treated individually rather than combined into a single index.



## 1.2 Data Granularity and Ethical Consideration

The OECD Regional Well-Being data is published at TL2 level as this level of granularity will protect individual privacy. If OECD were to publish the data at a more detailed level (say at the smaller TL3 levels), it will have to suppress dat to prevent potential identification of individuals or individual organisation. For example, at smaller TL3 levels, one might be able to identify a specific hospital when we look at mortality rate or make inference about the individual given the homicide rate. 

By using data granularity at TL2, this project works within the privacy protections already built into the data published by OECD.

However, as with all decisions, this may present a trade off in that TL2 regions are inherently large and internally diverse. For example, a seemingly wealthy region will contain residents with lower income vice versa. As such, the well-being indicators are at best an approximation of the real situation. 

## 1.3 Extraction of OECD Regional Well-Being Data from the Spreadsheet

In [5]:
from numpy import float64


column_names = [
    "country"
    , "region"
    , "region_code"
    , "disposable_income_pc" # 1. Disposable income per capita in USD PPP, 
    , "employment_rate" # 2. Employment rate, 
    , "unemployment_rate" # 3. Unemployment rate, 
    , "homicide_rate" # 4. Homicide rate per 100,000 people, 
    , "life_expectancy" # 5. Life expectancy (years), 
    , "secondary_education_pct" # 6. Population with at least secondary education (%age), 
    , "rooms_per_capita" # 7. Number of rooms per capita (rooms per person), 
    , "mortality_rate" # 8. Mortality rate (per 1000 people), 
    , "voter_turnout_pct" # 9. Voter turnout (%age), 
    , "broadband_access_pct" # 10. Households broadband access (%age of household), 
    , "air_quality_pm25" # 11. Air quality (PM2.5 in µg/m³), 
    , "life_satisfaction_index" # 12. Life satisfaction index, 
    , "housing_affordability_pct" # 13. Satisfaction with housing affordability (%age), 
    , "internet_speed_deviation" # 14. Internet speed from OECD average (%) and 
    , "social_network_support_index"# 15. Perceived social network support index
]

column_dtypes_mapping ={
    "country": str
    , "region": str
    , "region_code": str
    , "disposable_income_pc": float
    , "employment_rate": float
    , "unemployment_rate": float
    , "homicide_rate": float
    , "life_expectancy": float
    , "secondary_education_pct": float
    , "rooms_per_capita": float
    , "mortality_rate": float
    , "voter_turnout_pct": float
    , "broadband_access_pct": float
    , "air_quality_pm25": float
    , "life_satisfaction_index": float
    , "housing_affordability_pct": float
    , "internet_speed_deviation": float
    , "social_network_support_index": float
}

In [31]:
filename = "OECD-Regional-Well-Being-Data-File.xlsx"

df_regional_wellbeing_all_regions = pd.read_excel(
    f"{raw_dir}/{filename}"
    , sheet_name="Indicator_Last"
    , usecols="B:S"
    , skiprows=7
    , nrows=468 # row 476 - 8
    , names=column_names
    , dtype=column_dtypes_mapping 
)

print(f"Total number of rows: {len(df_regional_wellbeing_all_regions)}")


Total number of rows: 468


In [32]:
df_regional_wellbeing_all_regions.head(3)

,country,region,region_code,disposable_income_pc,employment_rate,unemployment_rate,homicide_rate,life_expectancy,secondary_education_pct,rooms_per_capita,mortality_rate,voter_turnout_pct,broadband_access_pct,air_quality_pm25,life_satisfaction_index,housing_affordability_pct,internet_speed_deviation,social_network_support_index
0,Australia,New South Wales,AU1,41725.3,73.6,4.2,1.0,83.2,85.8,2.3,3.0,90.7,85.2,9.7,7.2,35.3,-44.9,93.7
1,Australia,Victoria,AU2,36705.2,77.2,4.4,0.9,83.4,85.8,2.6,2.9,90.6,86.8,8.5,7.1,40.8,-41.5,93.8
2,Australia,Queensland,AU3,38043.9,77.5,3.8,0.8,82.7,83.9,2.3,3.1,88.2,86.3,8.0,7.0,47.8,-45.2,89.5


In [33]:
df_regional_wellbeing_all_regions.tail(3)

,country,region,region_code,disposable_income_pc,employment_rate,unemployment_rate,homicide_rate,life_expectancy,secondary_education_pct,rooms_per_capita,mortality_rate,voter_turnout_pct,broadband_access_pct,air_quality_pm25,life_satisfaction_index,housing_affordability_pct,internet_speed_deviation,social_network_support_index
465,Romania,Bucharest - Ilfov,RO32,37785.9,81.4,2.2,0.8,77.6,92.9,1.1,12.3,31.4,93.8,14.3,6.8,28.1,20.3,NaN
466,Romania,South West Oltenia,RO41,15619.7,50.8,6.8,0.7,77.0,79.7,1.3,12.0,35.5,85.9,10.5,6.4,46.5,-4.8,NaN
467,Romania,West,RO42,21600.4,55.8,4.1,0.1,76.5,86.5,1.1,12.9,29.9,90.5,10.6,6.5,45.9,10.9,NaN


## 1.4 Rudimentary Cleaning of the OECD Regional Well-Being Dataframe

📌 Keep the index column but rename it so that we can trace the row back to the original spreadsheet.

In [34]:
# keep the index and rename it. The index will allow me to trace the data back to the excel sheet.
df_regional_wellbeing_all_regions = df_regional_wellbeing_all_regions.reset_index()
df_regional_wellbeing_all_regions = df_regional_wellbeing_all_regions.rename(columns={"index": "xls_row_id"})

df_regional_wellbeing_all_regions.head(10)


,xls_row_id,country,region,region_code,disposable_income_pc,employment_rate,unemployment_rate,homicide_rate,life_expectancy,secondary_education_pct,rooms_per_capita,mortality_rate,voter_turnout_pct,broadband_access_pct,air_quality_pm25,life_satisfaction_index,housing_affordability_pct,internet_speed_deviation,social_network_support_index
0,0,Australia,New South Wales,AU1,41725.3,73.6,4.2,1.0,83.2,85.8,2.3,3.0,90.7,85.2,9.7,7.2,35.3,-44.9,93.7
1,1,Australia,Victoria,AU2,36705.2,77.2,4.4,0.9,83.4,85.8,2.6,2.9,90.6,86.8,8.5,7.1,40.8,-41.5,93.8
2,2,Australia,Queensland,AU3,38043.9,77.5,3.8,0.8,82.7,83.9,2.3,3.1,88.2,86.3,8.0,7.0,47.8,-45.2,89.5
3,3,Australia,South Australia,AU4,35469.9,71.7,4.0,0.6,83.0,80.2,2.3,3.0,91.1,82.5,6.1,7.1,49.8,-49.0,93.9
4,4,Australia,Western Australia,AU5,44015.1,74.7,3.8,0.8,83.6,83.9,2.6,2.8,88.0,88.4,7.6,7.1,52.0,-48.0,90.7
5,5,Australia,Tasmania,AU6,36216.5,76.2,4.8,0.5,82.1,78.1,2.6,3.4,92.4,83.4,5.3,7.4,28.8,-53.4,95.4
6,6,Australia,Northern Territory,AU7,50664.8,75.7,4.3,2.0,78.4,83.2,2.6,4.5,73.1,89.0,8.8,6.8,23.2,-47.4,83.3
7,7,Australia,Australian Capital Territory,AU8,65301.4,79.0,3.6,0.0,83.7,90.9,2.6,2.6,92.1,94.1,4.6,7.1,28.0,-46.4,91.4
8,8,Austria,Burgenland,AT11,38676.1,74.8,4.9,1.0,82.1,88.2,2.0,8.7,81.5,90.7,11.8,6.6,59.1,-52.7,89.1
9,9,Austria,Lower Austria,AT12,39242.3,71.3,4.0,0.8,81.5,87.7,1.9,9.0,80.6,90.7,12.0,7.1,55.0,-49.1,91.0


📌 Remove all preceding or trailing spaces on the 3 string columns.

In [35]:
# Make sure the string columns are cleaned
df_regional_wellbeing_all_regions['country'] = df_regional_wellbeing_all_regions['country'].str.strip()
df_regional_wellbeing_all_regions['region'] = df_regional_wellbeing_all_regions['region'].str.strip()
df_regional_wellbeing_all_regions['region_code'] = df_regional_wellbeing_all_regions['region_code'].str.strip()

📌 The country name will be converted to ISO3 country name convention using the handy `country_converter` package.w

In [36]:
import country_converter as coco

cc = coco.CountryConverter()

df_regional_wellbeing_all_regions["country_iso3"] = cc.pandas_convert(
    series=df_regional_wellbeing_all_regions['country'],
    to='ISO3'
)

df_regional_wellbeing_all_regions["country_iso3"].value_counts()

country_iso3
USA    51
COL    33
MEX    32
TUR    26
ITA    21
ESP    19
FRA    18
POL    17
CHL    16
DEU    16
NZL    14
CAN    13
GRC    13
NLD    12
GBR    12
JPN    10
LTU    10
AUT     9
PRT     9
AUS     8
CZE     8
HUN     8
SWE     8
ROU     8
KOR     7
CHE     7
CRI     6
ISR     6
NOR     6
BGR     6
DNK     5
EST     5
FIN     5
LVA     5
SVK     4
HRV     4
BEL     3
IRL     3
ISL     2
SVN     2
LUX     1
Name: count, dtype: int64

📌 Export the Data for Further Analysis

In [37]:
df_regional_wellbeing_all_regions.to_csv(f'{processed_dir}/01a_df_regional_wellbeing_all_regions.csv', index=False)
print(f"Exported: {processed_dir}/01a_df_regional_wellbeing_all_regions.csv")

Exported: ../data/processed/01a_df_regional_wellbeing_all_regions.csv


## 1.5 Extract Selected Countries for EDA and Analysis

As an MVP, the following 5 countries are selected as comparision (for the United Kingdom) countries for onward hypotheses testing, predictive modeling and some of the dashboarding presentation.

* France: 
  * Being the UK's nearest major neighbour across the Channel. 
  * It is also a good comparison country as it has its own regional disparities like the stuggling industrial region in the north and the relatively wealthy areas like Paris in the Île-de-France administrative region.
  * It has similar population size to the UK at 68m.
* Germany
  * Being the largest economy in Europe. 
  * The regions within Germany appears to have stronger identities within a structure where power is shared between a central national government and the regional government. 
  * This is in contrast to the United Kingdom where Scotland, Wales and Northern Ireland have devolved powers granted by the UK Parliament.
  * It has a population of 84m
* Italy
  * Italy is chosen for its stark North-South divide of its own. 
  * With Milan in the more prosperous Northern region and Campania in the poorer southern region, Italy provides a direct parallel to the UK.
  * The population of approxiately 59 million is in the same magitude as the UK.
  * Like the UK, Italy has a roughly long-narrow shape. 
* Spain 
  * Spain also has regional economic disparities with the likes of wealthier Madrid and the less wealthy Andalusia.
  * Spain also have distinct linguistic and cultural regions which could mean varying atttitudes towards well-being
  * The population of approxiately 48 million is in almost in same magitude as the UK.
  * Like the UK, Spain has a roughly long-narrow shape. 
* the Netherlands
  * The Netherlands is selected to be a defacto benchmark for what a more equitable and and regionally well-balance could look like.
  * It has a population of 18m.

In [39]:
selected_countries_iso3 = [
    "GBR"   # the Uk
    , "FRA" # France
    , "DEU" # Germany
    , "ITA" # Italy
    , "ESP" # Spain
    , "NLD" # the Netherlands
]

df_regional_wellbeing_6countries = df_regional_wellbeing_all_regions[df_regional_wellbeing_all_regions["country_iso3"].isin(selected_countries_iso3)]
df_regional_wellbeing_6countries["country"].value_counts()

country
Italy             21
Spain             19
France            18
Germany           16
Netherlands       12
United Kingdom    12
Name: count, dtype: int64

📌 Export the Data for Further Analysis

In [40]:
df_regional_wellbeing_6countries.to_csv(f'{processed_dir}/01b_df_regional_wellbeing_6countries.csv', index=False)
print(f"Exported: {processed_dir}/01b_df_regional_wellbeing_6countries.csv")

Exported: ../data/processed/01b_df_regional_wellbeing_6countries.csv


---

# 2.0 EDA of the OECD Regional Well-Being for the Chosen Six Countries

In [46]:
df_regional_wellbeing_6countries.shape

(98, 20)

In [41]:
df_regional_wellbeing_6countries.describe()

,xls_row_id,disposable_income_pc,employment_rate,unemployment_rate,homicide_rate,life_expectancy,secondary_education_pct,rooms_per_capita,mortality_rate,voter_turnout_pct,broadband_access_pct,air_quality_pm25,life_satisfaction_index,housing_affordability_pct,internet_speed_deviation,social_network_support_index
count,98.000000,98.000000,98.00000,98.000000,98.000000,98.000000,97.000000,97.000000,98.000000,98.000000,97.000000,98.000000,91.000000,91.000000,98.000000,92.000000
mean,230.387755,30157.332653,69.47449,7.389796,1.186735,82.190816,74.995876,1.822680,9.183673,69.042857,92.760825,10.973469,6.728571,49.615385,0.958163,90.529348
std,98.915317,5316.025577,10.51748,6.179122,2.224032,1.693108,10.368813,0.262006,2.198064,11.375403,7.356213,2.556195,0.423553,9.315995,29.071057,3.664518
min,111.000000,10472.200000,23.90000,1.900000,0.000000,74.900000,52.000000,1.000000,6.500000,36.200000,75.100000,5.900000,5.500000,26.600000,-82.000000,82.000000
25%,135.250000,26982.400000,65.07500,3.625000,0.600000,81.225000,68.000000,1.600000,7.700000,61.300000,86.500000,9.600000,6.400000,44.900000,-18.350000,88.500000
50%,191.500000,30325.500000,70.75000,5.300000,0.700000,82.200000,78.200000,1.900000,8.400000,70.000000,96.800000,10.900000,6.700000,49.800000,-2.200000,90.600000
75%,332.750000,33229.950000,77.15000,8.575000,1.000000,83.475000,82.500000,2.000000,9.300000,79.325000,98.600000,11.900000,7.000000,55.200000,19.400000,92.825000
max,398.000000,47301.700000,86.30000,37.500000,19.200000,85.400000,93.000000,2.300000,14.900000,84.800000,100.000000,21.900000,7.600000,75.500000,68.400000,100.000000


## 2.1 Missing Values

In [47]:
df_regional_wellbeing_6countries.isnull().sum()

xls_row_id                      0
country                         0
region                          0
region_code                     0
disposable_income_pc            0
employment_rate                 0
unemployment_rate               0
homicide_rate                   0
life_expectancy                 0
secondary_education_pct         1
rooms_per_capita                1
mortality_rate                  0
voter_turnout_pct               0
broadband_access_pct            1
air_quality_pm25                0
life_satisfaction_index         7
housing_affordability_pct       7
internet_speed_deviation        0
social_network_support_index    6
country_iso3                    0
dtype: int64

📌 Identify the rows with missing values so as to effectively address them.

In [49]:
df_rows_with_missing_values = df_regional_wellbeing_6countries[df_regional_wellbeing_6countries.isnull().any(axis=1)]

print(df_rows_with_missing_values)

     xls_row_id country         region region_code  disposable_income_pc  \
124         124  France     Guadeloupe        FRY1               24208.4   
125         125  France     Martinique        FRY2               26762.9   
126         126  France  French Guiana        FRY3               16260.1   
127         127  France     La Réunion        FRY4               26210.8   
128         128  France        Mayotte        FRY5               10472.2   
343         343   Spain          Ceuta        ES63               23767.3   
344         344   Spain        Melilla        ES64               22377.9   

     employment_rate  unemployment_rate  homicide_rate  life_expectancy  \
124             49.9               17.7            8.7             80.6   
125             58.6               10.8            5.6             81.7   
126             40.2               20.7           19.2             79.3   
127             48.3               15.9            1.5             82.3   
128             

📌 The rows with missing values are as follows: <a id="overseas_territories"></a>
  * French territories: Guadeloupe, Martinique, French Guiana, La Réunion, Mayotte (located in the Caribbean, South America, and Indian Ocean)
  * Spanish territories: Ceuta and Melilla (enclaves in North Africa)

These TL2 regions are not geographically part of mainland Europe and hence it is reasonable to assume that they are fundamentally different in socioeconomic contexts than the European countries that this project focusses. 

‼️ _**As such, these rows will not be included in the any of the analysis.**_

In [50]:
overseas_territories =[
    "Guadeloupe"
    , "Martinique"
    , "French Guiana"
    , "La Réunion"
    , "Mayotte"
    , "Ceuta"
    , "Melilla"]

In [52]:
df_regional_wellbeing_6countries_no_overseas = df_regional_wellbeing_6countries[
    ~df_regional_wellbeing_6countries["region"].isin(overseas_territories) # ~ and isin() means only rows that are NOT overseas territories
    ].reset_index(drop=True) # reset the index and drop the old one as the original index will have gaps


df_regional_wellbeing_6countries_no_overseas.describe()

,xls_row_id,disposable_income_pc,employment_rate,unemployment_rate,homicide_rate,life_expectancy,secondary_education_pct,rooms_per_capita,mortality_rate,voter_turnout_pct,broadband_access_pct,air_quality_pm25,life_satisfaction_index,housing_affordability_pct,internet_speed_deviation,social_network_support_index
count,91.000000,91.000000,91.000000,91.000000,91.000000,91.000000,91.000000,91.000000,91.000000,91.000000,91.000000,91.000000,91.000000,91.000000,91.000000,91.000000
mean,233.637363,30828.120879,71.238462,6.135165,0.848352,82.342857,75.921978,1.842857,9.170330,70.881319,93.126374,10.934066,6.728571,49.615385,-0.205495,90.543956
std,98.207662,4661.675464,8.120109,3.463247,0.799773,1.527972,9.798218,0.249507,2.226831,9.427960,7.124525,2.575578,0.423553,9.315995,27.361085,3.682125
min,111.000000,21391.700000,44.900000,1.900000,0.000000,79.200000,54.100000,1.300000,6.500000,50.800000,75.100000,5.900000,5.500000,26.600000,-53.900000,82.000000
25%,138.500000,28015.000000,66.150000,3.550000,0.600000,81.300000,69.450000,1.700000,7.700000,62.800000,86.650000,9.500000,6.400000,44.900000,-20.450000,88.500000
50%,193.000000,30498.400000,71.800000,5.100000,0.700000,82.300000,79.200000,1.900000,8.400000,70.700000,97.000000,10.900000,6.700000,49.800000,-4.100000,90.700000
75%,332.500000,33958.700000,77.250000,7.800000,1.000000,83.650000,83.150000,2.000000,9.300000,79.950000,98.650000,11.900000,7.000000,55.200000,18.200000,92.850000
max,398.000000,47301.700000,86.300000,19.300000,7.100000,85.400000,93.000000,2.300000,14.900000,84.800000,100.000000,21.900000,7.600000,75.500000,59.200000,100.000000


📌 Check that the missing value has been removed.

In [53]:
df_regional_wellbeing_6countries_no_overseas.isnull().sum()

xls_row_id                      0
country                         0
region                          0
region_code                     0
disposable_income_pc            0
employment_rate                 0
unemployment_rate               0
homicide_rate                   0
life_expectancy                 0
secondary_education_pct         0
rooms_per_capita                0
mortality_rate                  0
voter_turnout_pct               0
broadband_access_pct            0
air_quality_pm25                0
life_satisfaction_index         0
housing_affordability_pct       0
internet_speed_deviation        0
social_network_support_index    0
country_iso3                    0
dtype: int64

## 2.? Export the Data for Further Analysis

In [ ]:
df_regional_wellbeing_6countries_no_overseas.to_csv(f'{processed_dir}/01c_df_regional_wellbeing_6countries_no_overseas.csv', index=False)
print(f"Exported: {processed_dir}/01c_df_regional_wellbeing_6countries_no_overseas.csv")

Exported: ../data/processed/01b_df_regional_wellbeing_6countries.csv


---

# 3.0 Ethical Considerations

The selection of the six countries (the UK, France, Germany, Italy, Spain, Netherlands) within the Western European economies is predominantly a subjective choice. I have listed in section 1.5 of this notebook the main reasons for the choice which is mainly as a direct comparison for the UK.  

As such, the findings cannot be generalised to Eastern Europe or any other regions of the world and that with such a narrow selection of countries, certain perspectives might be missing.

Also, these countries have generally good clean data within the OECD dataset with the exception of the French and Spanish overseas territories([see Justification for removal of missing rows](#overseas_territories)). This is not surprising as these wealthy countries likely have better statistical collection endeavours and as such, a "availablity bias" has inevitably been introduced in this project where well-documented regions might be overly represented. 


---

# ?.0 Conclusions & Next Steps

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

---